# 🏍️ Two-Wheeler Parking Violation Detector — YOLOv8m Training (Free Colab T4)

This notebook trains a **YOLOv8m** model at **imgsz=1280** on a merged dataset of:
- Your custom annotations (exported from LabelImg/Roboflow)
- Roboflow Universe public datasets (motorcycle parking, Indian two-wheelers)

**Expected result after 100 epochs:** mAP50 ~0.82–0.88 on two-wheelers

## Steps
1. Run cells 1-3 to install and configure
2. Upload your dataset zip (from Roboflow export, YOLOv8 format)
3. Run cell 4 to merge datasets
4. Run cell 5 to train (~3-4 hours on free T4)
5. Download `best.pt` from the output
6. Place it in `data/trained_models/two_wheeler_detector.pt`

In [ ]:
# ── CELL 1: Install dependencies ──────────────────────────────────────────────
!pip install ultralytics roboflow -q
import os
from pathlib import Path
import shutil, yaml

# Check GPU
!nvidia-smi | head -5
print('\n✅ ultralytics installed')

In [ ]:
# ── CELL 2: Download public datasets from Roboflow Universe ──────────────────
# These are free public datasets — no API key needed for public access.
# Option A: Use Roboflow API (get free key at roboflow.com)
ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"  # <-- Replace with your free key

from roboflow import Roboflow

# Dataset 1: Motorcycle Parking (top-down & 45-degree angles)
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Indian two-wheeler parking dataset
try:
    project = rf.workspace("two-wheeler-parking").project("two-wheeler-parking-detection")
    ds1 = project.version(1).download("yolov8", location="/content/datasets/ds1")
    print('✅ Dataset 1 downloaded')
except Exception as e:
    print(f'Dataset 1 skipped: {e}')

# Motorcycle parking lot detection
try:
    project2 = rf.workspace("motorcycle-parking").project("motorcycle-parking-lot")
    ds2 = project2.version(1).download("yolov8", location="/content/datasets/ds2")
    print('✅ Dataset 2 downloaded')
except Exception as e:
    print(f'Dataset 2 skipped: {e}')

print('\nNote: If API key not available, skip to Cell 2b for manual upload')

In [ ]:
# ── CELL 2b: Manual upload of YOUR dataset (skip if Cell 2 worked) ───────────
# Upload your dataset zip from Roboflow (YOLOv8 format) via Files panel on left,
# then unzip it here.
from google.colab import files
print('Upload your dataset zip from Roboflow (YOLOv8 format)...')
uploaded = files.upload()  # Select your .zip file

import zipfile
for fname in uploaded:
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall('/content/datasets/my_dataset')
    print(f'✅ Extracted {fname} → /content/datasets/my_dataset/')

# List contents
!find /content/datasets/my_dataset -name '*.yaml' | head -5
!find /content/datasets/my_dataset -name '*.jpg' | wc -l

In [ ]:
# ── CELL 3: Merge all datasets into one unified dataset ──────────────────────
import os, shutil, random, yaml
from pathlib import Path

MERGED = Path('/content/merged_dataset')
for split in ['train', 'val']:
    (MERGED / split / 'images').mkdir(parents=True, exist_ok=True)
    (MERGED / split / 'labels').mkdir(parents=True, exist_ok=True)

# Class mapping — we want: 0=motorcycle, 1=bicycle
# Adjust these if your datasets use different class orders
CANONICAL_CLASSES = ['motorcycle', 'bicycle']

def merge_dataset(src_dir, class_map=None):
    """Copy images+labels from src_dir into MERGED, remapping classes."""
    src = Path(src_dir)
    copied = 0
    for split in ['train', 'valid', 'val']:
        img_dir = src / split / 'images'
        lbl_dir = src / split / 'labels'
        dest_split = 'train' if split == 'train' else 'val'
        if not img_dir.exists():
            continue
        for img in img_dir.glob('*.*'):
            lbl = lbl_dir / (img.stem + '.txt')
            if not lbl.exists():
                continue
            # Copy image
            dst_img = MERGED / dest_split / 'images' / img.name
            shutil.copy2(img, dst_img)
            # Read and remap labels
            lines = lbl.read_text().strip().splitlines()
            new_lines = []
            for line in lines:
                parts = line.split()
                if not parts:
                    continue
                orig_cls = int(parts[0])
                if class_map and orig_cls in class_map:
                    new_cls = class_map[orig_cls]
                else:
                    new_cls = 0  # default to motorcycle
                new_lines.append(f"{new_cls} " + " ".join(parts[1:]))
            (MERGED / dest_split / 'labels' / (img.stem + '.txt')).write_text('\n'.join(new_lines))
            copied += 1
    print(f'  Copied {copied} images from {src_dir}')
    return copied

total = 0
for ds_dir in Path('/content/datasets').iterdir():
    if ds_dir.is_dir():
        # Find the yaml to read class names
        yaml_files = list(ds_dir.glob('*.yaml')) + list(ds_dir.glob('**/*.yaml'))
        class_map = None
        if yaml_files:
            ds_yaml = yaml.safe_load(yaml_files[0].read_text())
            ds_names = ds_yaml.get('names', [])
            # Build class_map: ds class index -> canonical index
            class_map = {}
            for i, name in enumerate(ds_names):
                n = name.lower()
                if any(k in n for k in ['motor', 'bike', 'scooter', 'moped']):
                    class_map[i] = 0  # → motorcycle
                elif any(k in n for k in ['cycle', 'bicycle', 'pedal']):
                    class_map[i] = 1  # → bicycle
        total += merge_dataset(str(ds_dir), class_map)

# Write merged data.yaml
merged_yaml = {
    'path': str(MERGED),
    'train': 'train/images',
    'val': 'val/images',
    'nc': 2,
    'names': CANONICAL_CLASSES,
}
(MERGED / 'data.yaml').write_text(yaml.dump(merged_yaml))

train_imgs = list((MERGED / 'train' / 'images').glob('*.*'))
val_imgs   = list((MERGED / 'val'   / 'images').glob('*.*'))
print(f'\n✅ Merged dataset ready:')
print(f'   Train: {len(train_imgs)} images')
print(f'   Val:   {len(val_imgs)} images')
print(f'   Classes: {CANONICAL_CLASSES}')

In [ ]:
# ── CELL 4: Train YOLOv8m ─────────────────────────────────────────────────────
from ultralytics import YOLO

model = YOLO('yolov8m.pt')  # YOLOv8 Medium — good accuracy/speed tradeoff

results = model.train(
    data='/content/merged_dataset/data.yaml',
    epochs=100,
    imgsz=1280,        # High resolution for small objects in parking lots
    batch=8,           # T4 16GB can handle batch=8 at imgsz=1280
    device=0,          # GPU
    project='/content/runs',
    name='two_wheeler_v1',
    exist_ok=True,
    patience=20,       # Early stopping
    # Heavy augmentation for diverse parking scenarios
    fliplr=0.5,        # Horizontal flip
    flipud=0.1,        # Vertical flip (for top-down images)
    scale=0.6,         # Scale variation
    mosaic=1.0,        # Mosaic augmentation
    hsv_h=0.015,       # Hue variation
    hsv_s=0.7,         # Saturation variation
    hsv_v=0.4,         # Brightness variation
    degrees=10.0,      # Rotation (parking lot camera angles)
    translate=0.1,     # Translation
    copy_paste=0.3,    # Copy-paste augmentation (good for parking lots)
    workers=2,
    verbose=True,
)

print('\n✅ Training complete!')
print(f'Best weights: /content/runs/two_wheeler_v1/weights/best.pt')
print(f'mAP50: {results.results_dict.get("metrics/mAP50(B)", "N/A")}')
print(f'mAP50-95: {results.results_dict.get("metrics/mAP50-95(B)", "N/A")}')

In [ ]:
# ── CELL 5: Validate and Download ─────────────────────────────────────────────
from ultralytics import YOLO
from google.colab import files

model = YOLO('/content/runs/two_wheeler_v1/weights/best.pt')

# Validate on val set
val_results = model.val(data='/content/merged_dataset/data.yaml', imgsz=1280)
print(f'\nValidation Results:')
print(f'  mAP50:     {val_results.box.map50:.4f}')
print(f'  mAP50-95:  {val_results.box.map:.4f}')
print(f'  Precision: {val_results.box.mp:.4f}')
print(f'  Recall:    {val_results.box.mr:.4f}')

# Download best.pt
print('\nDownloading best.pt ...')
files.download('/content/runs/two_wheeler_v1/weights/best.pt')
print('\n📋 NEXT STEP:')
print('  1. Rename the downloaded file to: two_wheeler_detector.pt')
print('  2. Place it in: data/trained_models/two_wheeler_detector.pt')
print('  3. Restart the backend server')

In [ ]:
# ── CELL 6 (Optional): Test on a sample image ─────────────────────────────────
from google.colab import files
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt

model = YOLO('/content/runs/two_wheeler_v1/weights/best.pt')

# Upload a test image
print('Upload a parking lot test image...')
up = files.upload()
test_img = list(up.keys())[0]

results = model(test_img, conf=0.30, imgsz=1280)
annotated = results[0].plot()

plt.figure(figsize=(14, 10))
plt.imshow(annotated[:, :, ::-1])  # BGR → RGB
plt.axis('off')
plt.title(f'Detected: {len(results[0].boxes)} vehicles (conf >= 30%)')
plt.tight_layout()
plt.savefig('/content/test_output.jpg', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved test output to /content/test_output.jpg')